# 전이 데이터셋 생성

`two_visit_dataset.json`과 `multi_visit_dataset.json`을 통합하고,
검진 간격 필터를 적용한 뒤 `adoc_v1.total_checkups.json`의 원본 검진 필드를 결합하여
**pre-diabetes** / **diabetes** 학습용 데이터셋을 생성합니다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from core.dataset_builder import TransitionDatasetBuilder

OUTPUT_DIR = Path("../outputs/260526_create_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDA_OUTPUT_DIR = Path("../outputs/260522_EDA")

# current_checkup_date → future_checkup_date 간격 상한 (년)
# label=1: 이 값 이하인 레코드만 유지
MAX_INTERVAL_YEARS = 1.0

# label=0의 간격 상한 여유 (년)
# label=0: MAX_INTERVAL_YEARS 초과 AND MAX_INTERVAL_YEARS + NEGATIVE_BUFFER_YEARS 이하인 레코드만 유지
NEGATIVE_BUFFER_YEARS = 0.5 

EXCLUDE_USER_KEYS = [
    "INVALID_RESULT",
]

# 해당 접두사로 시작하는 user_key를 가진 레코드 제외 (익명·워크인 수검자)
EXCLUDE_USER_KEY_PREFIXES = [
    "ANONYMOUS",
    "WALKIN",
]

## 1. 데이터셋 생성

`two_visit_dataset.json` (2회 수검) + `multi_visit_dataset.json` (3회+ 수검)을 통합한 뒤,  
아래 필터를 순서대로 적용하여 최종 학습용 데이터셋을 생성합니다.

### 필터링 규칙

**Step 0. 익명·워크인 수검자 제외**
- `user_key`가 `ANONYMOUS` 또는 `WALKIN`으로 시작하는 레코드 제외

**Step 1. 검진 간격 필터 + 국가검진 시작 제외** (label 기준 반전 적용)
- **label=1**: 간격이 `MAX_INTERVAL_YEARS` **이하**인 레코드만 유지
- **label=0**: 간격이 `MAX_INTERVAL_YEARS` **초과** AND `MAX_INTERVAL_YEARS + NEGATIVE_BUFFER_YEARS` **이하**인 레코드만 유지
  - 충분한 추적 기간 동안 전이가 없었음을 확인하되, 관측 윈도우를 제한하여 label=1과 시간 범위를 맞춤
- `selected_transition`의 첫 번째 상태(현재 검진)에 `(국가)`가 포함된 레코드 제외

**Step 2. 당뇨병용제 처방 이력 필터** (label=0만 적용)
- label=0 레코드 중, `current_checkup_date + MAX_INTERVAL_YEARS` 이전 시점에 당뇨병용제 처방 이력이 **하나라도** 있는 경우 제외
  - 현재 검진 이전 과거 처방 이력이 있는 경우에도 당뇨 진행 가능성이 있어 음성 레이블로 보기 어렵기 때문

**Step 3. 중복 제거** (동일 dataset 내 user_key는 반드시 유일)
- 동일 user_key에 유효한 전이 흐름이 여러 개인 경우
  - label=0 후보가 여럿: 이후 검진일의 공복혈당이 **가장 낮은** 검진일 선택
  - label=1 후보가 여럿: 이후 검진일의 공복혈당이 **가장 높은** 검진일 선택
- label=0 과 label=1 후보가 동시에 존재하는 경우: **label=1 우선 보존**

---

결과는 `detail_infos`를 `small_checkup_name` → `value` 형태로 펼쳐  
`pre_diabetes_dataset.xlsx`, `diabetes_dataset.xlsx`로 저장합니다.

In [ ]:
builder = TransitionDatasetBuilder(
    EDA_OUTPUT_DIR / "two_visit_dataset.json",
    EDA_OUTPUT_DIR / "multi_visit_dataset.json",
    max_interval_years=MAX_INTERVAL_YEARS,
    negative_buffer_years=NEGATIVE_BUFFER_YEARS,
    exclude_user_keys=EXCLUDE_USER_KEYS,
    exclude_user_key_prefixes=EXCLUDE_USER_KEY_PREFIXES,
)

datasets = builder.build()
saved_paths = builder.export(OUTPUT_DIR)

In [ ]:
builder.print_filter_stats()

════════════════════════════════════════════════════════════════════════
                                pre-diabetes      diabetes            합계
════════════════════════════════════════════════════════════════════════
원천 레코드 (label=0)                    198,468건      114,208건      312,676건
원천 레코드 (label=1)                     13,871건        1,753건       15,624건
원천 합계                               212,339건      115,961건      328,300건
────────────────────────────────────────────────────────────────────────
  [Step 1] 검진 간격 필터  (label=1: ≤1.0년 / label=0: 1.0~1.5년)
  label=0 간격 부족 제외                   25,158건       14,337건       39,495건
  label=0 간격 초과 제외                  151,252건       87,980건      239,232건
  label=1 간격 초과 제외                   10,521건        1,360건       11,881건
  (국가)검진 시작 제외                       23,902건       11,540건       35,442건
  검진 레코드 미매칭                              2건            0건            2건
──────────────────────────────────────────────────────────────────

## 2. 요약

In [ ]:
summary = builder.summary()
summary["mean_interval_days"] = summary["mean_interval_days"].round(1)
summary

,dataset,label,source,count,mean_interval_days
0,diabetes,0,2회,195,409.1
1,diabetes,0,3회+,405,414.1
2,diabetes,1,2회,15,316.9
3,diabetes,1,3회+,14,341.6
4,pre-diabetes,0,2회,360,412.8
5,pre-diabetes,0,3회+,651,417.4
6,pre-diabetes,1,2회,92,311.1
7,pre-diabetes,1,3회+,137,314.2


In [ ]:
builder.print_label_distribution("pre-diabetes")
builder.print_label_distribution("diabetes")

── pre-diabetes ──
  shape : (1240, 339)
  label 0 :  1011개  (81.5%)
  label 1 :   229개  (18.5%)

── diabetes ──
  shape : (629, 317)
  label 0 :   600개  (95.4%)
  label 1 :    29개  (4.6%)

